In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, BatchNormalization, Activation, MaxPooling3D, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l2

# Load the training dataset
X_train = np.load('/path/to/training/data/X_train.npy')  # Replace with actual path
Y_train = np.load('/path/to/training/data/Y_train.npy')  # Replace with actual path

# Add a depth dimension (assuming channel-last format)
X_train = np.expand_dims(X_train, axis=-2)  # Expanding to (529, 26, 21, 1, 3)

# Define a standard 3D CNN with strong regularization
model = Sequential([
    Conv3D(32, kernel_size=(3, 3, 3), padding="same", kernel_regularizer=l2(0.01), input_shape=X_train.shape[1:]),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling3D(pool_size=(2, 2, 1)),

    Conv3D(64, kernel_size=(3, 3, 3), padding="same", kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling3D(pool_size=(2, 2, 1)),

    Conv3D(128, kernel_size=(3, 3, 3), padding="same", kernel_regularizer=l2(0.01)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling3D(pool_size=(2, 2, 1)),

    Flatten(),
    Dense(256, activation="relu", kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    Dense(128, activation="relu", kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    Dense(1, activation="linear")  # Assuming regression problem
])

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss="mean_squared_error",
              metrics=["mae"])

# Train the model only on the training dataset
model.fit(X_train, Y_train, epochs=50, batch_size=16, verbose=1)

# Save the model
model.save("3D_CNN_trained_model.h5")


In [ ]:
import os
import tensorflow as tf

# Load the saved model
model_path = "3D_CNN_trained_model.h5"  # Make sure this matches your saved model name
model = tf.keras.models.load_model(model_path)

# Print model summary (statistics)
print("✅ Model Summary:")
model.summary()

# Print directory of saved model
abs_path = os.path.abspath(model_path)
print(f"\n✅ Model saved at: {abs_path}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load the validation dataset
X_val = np.load("/path/to/validation/data/X_val.npy")  # Shape: (157, 26, 21, 3)
Y_val = np.load("/path/to/validation/data/Y_val.npy")  # Shape: (157,)

# Print original shape
print(f"Original X_val shape: {X_val.shape}")

# Reshape X_val to match the expected shape (None, 26, 21, 1, 3)
X_val = np.expand_dims(X_val, axis=3)  # Add a depth dimension at axis=3

# Print new shape after modification
print(f"Modified X_val shape: {X_val.shape}")

# Load the trained model
model = tf.keras.models.load_model("/path/to/trained/3D_CNN_trained_model.h5")

# Print expected input shape
expected_input_shape = model.input_shape
print(f"Model expected input shape: {expected_input_shape}")

# Predict Y values using the model
Y_pred = model.predict(X_val)

# Compute the squared error cost function
squared_errors = np.square(Y_pred.flatten() - Y_val)  # Flatten predictions and compute squared differences
squared_error_cost = np.mean(squared_errors)  # Calculate mean squared error

# Print results
print(f"✅ Squared Error Cost Function on Validation Dataset: {squared_error_cost:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf

# Load the trained model
model_path = "/path/to/trained/3D_CNN_trained_model.h5"  # Update path if needed
model = tf.keras.models.load_model(model_path)

# Load training dataset
X_train = np.load("/path/to/training/data/X_train.npy")  # Shape: (529, 26, 21, 3)
Y_train = np.load("/path/to/training/data/Y_train.npy")  # Shape: (529,)

# Add depth dimension: Reshape X_train from (N, 26, 21, 3) → (N, 26, 21, 1, 3)
X_train = np.expand_dims(X_train, axis=3)  # Now (529, 26, 21, 1, 3)
print(f"✅ Adjusted X_train shape: {X_train.shape}")  # Debugging

# Predict Y values using the model
Y_pred = model.predict(X_train)

# Ensure Y_train has the correct shape
Y_train = Y_train.reshape(-1, 1)  # Ensure Y_train has shape (N, 1)

# Compute squared error cost function
squared_error = np.square(Y_pred - Y_train)
mean_squared_error = np.mean(squared_error)

# Print results
print(f"✅ Mean Squared Error on Training Dataset: {mean_squared_error:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error

# Load the trained model
model_path = "/path/to/trained/3D_CNN_trained_model.h5"  # Update path if needed
model = tf.keras.models.load_model(model_path)

# Load testing dataset
X_test = np.load("/path/to/testing/data/X_test.npy")  # Shape: (157, 26, 21, 3)
Y_test = np.load("/path/to/testing/data/Y_test.npy")  # Shape: (157,)

# Add depth dimension: Reshape X_test from (N, 26, 21, 3) → (N, 26, 21, 1, 3)
X_test = np.expand_dims(X_test, axis=3)  # Now (157, 26, 21, 1, 3)
print(f"✅ Adjusted X_test shape: {X_test.shape}")  # Debugging

# Predict Y values using the model
Y_pred = model.predict(X_test)

# Ensure Y_test has the correct shape
Y_test = Y_test.reshape(-1, 1)  # Ensure Y_test has shape (N, 1)

# Compute Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))

# Print results
print(f"✅ Root Mean Squared Error (RMSE) on Testing Dataset: {rmse:.6f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

# Load the trained model
model_path = "/path/to/trained/3D_CNN_trained_model.h5"  # Update path if needed
model = tf.keras.models.load_model(model_path)

# Load testing dataset
X_test = np.load("/path/to/testing/data/X_test.npy")  # Shape: (157, 26, 21, 3)
Y_test = np.load("/path/to/testing/data/Y_test.npy")  # Shape: (157,)

# Add depth dimension: Reshape X_test from (N, 26, 21, 3) → (N, 26, 21, 1, 3)
X_test = np.expand_dims(X_test, axis=3)  # Now (157, 26, 21, 1, 3)
print(f"✅ Adjusted X_test shape: {X_test.shape}")  # Debugging

# Predict Y values using the model
Y_pred = model.predict(X_test)

# Ensure Y_test has the correct shape
Y_test = Y_test.reshape(-1, 1)  # Ensure Y_test has shape (N, 1)

# Compute R² Score
r2 = r2_score(Y_test, Y_pred)

print(f"Coefficient of Determination (R²) on Testing Set: {r2:.4f}")
